In [4]:
!pip install PySastrawi pandas numpy

  Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.4.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.1-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 180.2 kB/s  0:01:25m0:00:0100:04
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


# Tugas STKI: Perbandingan TF-IDF dan VSM

Notebook ini membandingkan dua metode perankingan dokumen:
- **TF-IDF**: penjumlahan skor bobot term query pada setiap dokumen.
- **VSM**: cosine similarity antara vektor query dan vektor dokumen berbasis TF-IDF.

Semua komentar, label, dan output menggunakan Bahasa Indonesia.

## Teori Singkat Sesuai Materi Kuliah

Sistem temu kembali informasi (Information Retrieval/IR) adalah sistem yang menerima masukan berupa query dari pengguna, memproses query tersebut terhadap koleksi dokumen, lalu mengembalikan dokumen yang dianggap relevan. Relevansi biasanya ditentukan dari tingkat kemiripan antara isi query dan isi dokumen. Dengan demikian, tujuan utama IR adalah membantu pengguna menemukan informasi yang paling sesuai secara cepat.

Dalam IR, terdapat beberapa model representasi. **Boolean model** (set-theoretic) memandang dokumen sebagai relevan atau tidak relevan secara biner (0/1) menggunakan operator logika **AND**, **OR**, dan **NOT**. Sementara itu, **Vector Space Model** (algebraic) merepresentasikan query dan dokumen sebagai vektor di ruang berdimensi term, kemudian menghitung derajat kemiripannya.

Pada notebook ini, implementasi difokuskan pada pembobotan term menggunakan **TF-IDF** dan proses perankingan menggunakan **Vector Space Model** dengan **cosine similarity** untuk menghasilkan urutan dokumen paling relevan terhadap query.

## Install Library

Library inti sudah diinstal pada cell pertama menggunakan `PySastrawi`.

## Import & Konfigurasi

In [1]:
import math
import re
from collections import Counter

import numpy as np
import pandas as pd

from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

print('Import dan konfigurasi selesai. Teks tabel tidak dipotong.')

Import dan konfigurasi selesai. Teks tabel tidak dipotong.


## Rumus TF, IDF, dan TF-IDF

Jenis TF yang umum digunakan pada materi:
- **TF biner**: nilai TF hanya 0 atau 1, tergantung term muncul atau tidak pada dokumen.
- **TF murni (raw TF)**: nilai TF adalah frekuensi kemunculan term secara langsung.
- **TF logaritmik**: nilai TF diperkecil secara log agar frekuensi sangat besar tidak terlalu dominan.
- **TF normalisasi**: nilai TF dibagi total term dalam dokumen agar panjang dokumen lebih terkendali.

Rumus yang digunakan:
- TF_log = 0 jika tf = 0, selain itu TF_log = 1 + log10(tf)
- IDF = log10(D / df), dengan D = jumlah dokumen, df = jumlah dokumen yang mengandung term
- Bobot TF-IDF: w_(i,j) = TF_(i,j) * IDF_j

Secara intuitif, semakin sering kata muncul dalam suatu dokumen maka nilai TF cenderung makin besar. Sebaliknya, semakin jarang kata muncul di seluruh koleksi maka nilai IDF makin besar, sehingga kombinasi TF-IDF menonjolkan kata yang penting dan spesifik.

In [8]:
# Contoh mini: perbedaan TF raw vs TF log untuk satu term
term_contoh = 'digital'
tf_raw_contoh = 5
D_contoh = 5
df_contoh = 2

tf_log_contoh = 0 if tf_raw_contoh == 0 else 1 + math.log10(tf_raw_contoh)
idf_contoh = math.log10(D_contoh / df_contoh)

bobot_raw = tf_raw_contoh * idf_contoh
bobot_log = tf_log_contoh * idf_contoh

print('Contoh mini TF raw vs TF log (satu term):')
print(f'Term      : {term_contoh}')
print(f'TF_raw    : {tf_raw_contoh}')
print(f'TF_log    : {tf_log_contoh:.6f}')
print(f'IDF       : {idf_contoh:.6f}')
print(f'Bobot_raw : {bobot_raw:.6f}')
print(f'Bobot_log : {bobot_log:.6f}')
# TF log membantu mengurangi dominasi term yang frekuensinya sangat tinggi dibanding TF murni.

Contoh mini TF raw vs TF log (satu term):
Term      : digital
TF_raw    : 5
TF_log    : 1.698970
IDF       : 0.397940
Bobot_raw : 1.989700
Bobot_log : 0.676088


## Dokumen Korpus

Korpus berisi 5 dokumen pendek Bahasa Indonesia dengan topik berbeda.

In [2]:
dokumen = {
    'D1': 'Perusahaan teknologi mengembangkan kecerdasan buatan untuk analisis data pelanggan dan otomatisasi layanan digital di kota besar.',
    'D2': 'Tim sepak bola nasional menjalani latihan intensif untuk menghadapi turnamen regional dan meningkatkan ketahanan fisik pemain.',
    'D3': 'Sekolah menengah mulai menerapkan platform pembelajaran digital berbasis kecerdasan buatan untuk membantu evaluasi siswa.',
    'D4': 'Puskesmas mengadakan program edukasi kesehatan ibu dan anak serta kampanye gizi seimbang di wilayah pedesaan.',
    'D5': 'Komunitas lingkungan menanam mangrove di pesisir untuk mengurangi abrasi dan menjaga ekosistem laut tetap stabil.'
}

df_korpus = pd.DataFrame({
    'Dokumen': list(dokumen.keys()),
    'Teks Asli': list(dokumen.values())
})

display(df_korpus)

,Dokumen,Teks Asli
0,D1,Perusahaan teknologi mengembangkan kecerdasan buatan untuk analisis data pelanggan dan otomatisasi layanan digital di kota besar.
1,D2,Tim sepak bola nasional menjalani latihan intensif untuk menghadapi turnamen regional dan meningkatkan ketahanan fisik pemain.
2,D3,Sekolah menengah mulai menerapkan platform pembelajaran digital berbasis kecerdasan buatan untuk membantu evaluasi siswa.
3,D4,Puskesmas mengadakan program edukasi kesehatan ibu dan anak serta kampanye gizi seimbang di wilayah pedesaan.
4,D5,Komunitas lingkungan menanam mangrove di pesisir untuk mengurangi abrasi dan menjaga ekosistem laut tetap stabil.


## Preprocessing

Pipeline `preprocess(text)` (wajib urut):
1. Case folding
2. Cleaning (regex: hapus angka dan tanda baca)
3. Tokenizing
4. Stopword Removal (`StopWordRemoverFactory`)
5. Stemming (`StemmerFactory`)

In [3]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = text.split()
    teks_tanpa_stopword = stopword_remover.remove(' '.join(tokens))
    tokens_tanpa_stopword = teks_tanpa_stopword.split()
    tokens_stem = [stemmer.stem(token) for token in tokens_tanpa_stopword]
    return tokens_stem

hasil_preproses = {nama_dok: preprocess(teks) for nama_dok, teks in dokumen.items()}

df_preproses = pd.DataFrame({
    'Dokumen': list(hasil_preproses.keys()),
    'Token Hasil Preprocessing': [' '.join(v) for v in hasil_preproses.values()]
})

display(df_preproses)

,Dokumen,Token Hasil Preprocessing
0,D1,usaha teknologi kembang cerdas buat analisis data langgan otomatisasi layan digital kota
1,D2,tim sepak bola nasional jalan latih intensif hadap turnamen regional tingkat tahan fisik main
2,D3,sekolah tengah terap platform ajar digital bas cerdas buat bantu evaluasi siswa
3,D4,puskesmas ada program edukasi sehat anak kampanye gizi imbang wilayah desa
4,D5,komunitas lingkung tanam mangrove pesisir kurang abrasi jaga ekosistem laut stabil


## Perhitungan TF-IDF Manual

Perhitungan dilakukan manual tanpa `sklearn/TfidfVectorizer`.

- TF(d, t) = frekuensi term `t` pada dokumen `d` / total token dokumen `d`
- IDF(t) = log10(N / df(t))
- TF-IDF(d, t) = TF(d, t) × IDF(t)

Output utama section ini adalah tabel lengkap **term × dokumen**.

In [4]:
nama_dokumen = list(hasil_preproses.keys())
N = len(nama_dokumen)

vocab = sorted({term for tokens in hasil_preproses.values() for term in tokens})

# Query ditentukan di sini agar bisa ikut ke tabel TF + DF + IDF
query = 'kecerdasan buatan pendidikan digital'
query_tokens = preprocess(query)
query_counter = Counter(query_tokens)

# TF mentah (jumlah kemunculan) untuk tampilan tabel komponen
# Kolom: Q, D1, D2, ... + df + D/df + IDF log(D/df)
tf_raw = {dok: Counter(tokens) for dok, tokens in hasil_preproses.items()}

df = {
    term: sum(1 for dok in nama_dokumen if tf_raw[dok].get(term, 0) > 0)
    for term in vocab
}

rasio_D_df = {term: (N / df[term]) if df[term] > 0 else 0.0 for term in vocab}
idf = {term: math.log10(rasio_D_df[term]) if df[term] > 0 else 0.0 for term in vocab}

df_komponen = pd.DataFrame(index=vocab)
df_komponen['Q'] = [query_counter.get(term, 0) for term in vocab]
for dok in nama_dokumen:
    df_komponen[dok] = [tf_raw[dok].get(term, 0) for term in vocab]
df_komponen['df'] = [df[term] for term in vocab]
df_komponen['D/df'] = [rasio_D_df[term] for term in vocab]
df_komponen['IDF log(D/df)'] = [idf[term] for term in vocab]
df_komponen.index.name = 'Token'

print('Tabel komponen perhitungan (TF mentah + DF + D/df + IDF):')
display(df_komponen.round(3))

# TF ternormalisasi untuk perhitungan TF-IDF
# TF(d, t) = frekuensi term t pada dokumen d / total token dokumen d
tf = {}
for dok, tokens in hasil_preproses.items():
    counter = Counter(tokens)
    total_token = len(tokens) if len(tokens) > 0 else 1
    tf[dok] = {term: counter.get(term, 0) / total_token for term in vocab}

# TF-IDF manual
tfidf = {
    dok: {term: tf[dok][term] * idf[term] for term in vocab}
    for dok in nama_dokumen
}

# Tabel TF-IDF lengkap: term x dokumen
df_tfidf = pd.DataFrame(
    {dok: [tfidf[dok][term] for term in vocab] for dok in nama_dokumen},
    index=vocab
)
df_tfidf.index.name = 'Term'

print('Tabel TF-IDF lengkap (term x dokumen):')
display(df_tfidf.round(6))

Tabel komponen perhitungan (TF mentah + DF + D/df + IDF):


,Q,D1,D2,D3,D4,D5,df,D/df,IDF log(D/df)
Token,,,,,,,,,
abrasi,0,0,0,0,0,1,1,5.0,0.699
ada,0,0,0,0,1,0,1,5.0,0.699
ajar,0,0,0,1,0,0,1,5.0,0.699
anak,0,0,0,0,1,0,1,5.0,0.699
analisis,0,1,0,0,0,0,1,5.0,0.699
bantu,0,0,0,1,0,0,1,5.0,0.699
bas,0,0,0,1,0,0,1,5.0,0.699
bola,0,0,1,0,0,0,1,5.0,0.699
buat,1,1,0,1,0,0,2,2.5,0.398


Tabel TF-IDF lengkap (term x dokumen):


,D1,D2,D3,D4,D5
Term,,,,,
abrasi,0.000000,0.000000,0.000000,0.000000,0.063543
ada,0.000000,0.000000,0.000000,0.063543,0.000000
ajar,0.000000,0.000000,0.058248,0.000000,0.000000
anak,0.000000,0.000000,0.000000,0.063543,0.000000
analisis,0.058248,0.000000,0.000000,0.000000,0.000000
bantu,0.000000,0.000000,0.058248,0.000000,0.000000
bas,0.000000,0.000000,0.058248,0.000000,0.000000
bola,0.000000,0.049926,0.000000,0.000000,0.000000
buat,0.033162,0.000000,0.033162,0.000000,0.000000


## Ranking dengan TF-IDF

Skor dokumen dihitung dengan menjumlahkan bobot TF-IDF term-term query pada setiap dokumen.

In [5]:
query = 'kecerdasan buatan pendidikan digital'
query_tokens = preprocess(query)
query_terms = [term for term in query_tokens if term in vocab]

skor_tfidf = {}
for dok in nama_dokumen:
    skor_tfidf[dok] = sum(tfidf[dok][term] for term in query_terms)

ranking_tfidf = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Skor TF-IDF': [skor_tfidf[d] for d in nama_dokumen]
}).sort_values('Skor TF-IDF', ascending=False).reset_index(drop=True)

ranking_tfidf['Ranking TF-IDF'] = ranking_tfidf.index + 1

print('Query:', query)
print('Term query setelah preprocessing:', query_terms)
display(ranking_tfidf)

Query: kecerdasan buatan pendidikan digital
Term query setelah preprocessing: ['cerdas', 'buat', 'digital']


,Dokumen,Skor TF-IDF,Ranking TF-IDF
0,D1,0.099485,1
1,D3,0.099485,2
2,D2,0.000000,3
3,D4,0.000000,4
4,D5,0.000000,5


## Ranking dengan VSM (Cosine Similarity)

Representasikan query dan dokumen sebagai vektor TF-IDF, kemudian hitung cosine similarity.

Langkah perhitungan VSM sesuai materi:
1. Menghitung bobot tf-idf untuk setiap term di setiap dokumen dan query.
2. Menghitung panjang vektor (norma) untuk query dan tiap dokumen.
3. Menghitung dot product antara vektor query dan vektor dokumen.
4. Menghitung nilai cosine similarity = (Q · D) / (|Q| * |D|).
5. Mengurutkan dokumen berdasarkan nilai cosine similarity (ranking).

In [ ]:
def cosine_similarity_manual(vec_a, vec_b):
    # Dot product antara vektor query dan vektor dokumen
    pembilang = float(np.dot(vec_a, vec_b))
    # Norm vektor query dan dokumen, lalu dikalikan sebagai penyebut
    penyebut = float(np.linalg.norm(vec_a) * np.linalg.norm(vec_b))
    # Nilai cosine similarity
    return pembilang / penyebut if penyebut != 0 else 0.0

counter_query = Counter(query_terms)
total_query = len(query_terms) if len(query_terms) > 0 else 1

tf_query = {term: counter_query.get(term, 0) / total_query for term in vocab}
vektor_query = np.array([tf_query[term] * idf[term] for term in vocab], dtype=float)

skor_vsm = {}
for dok in nama_dokumen:
    vektor_dok = np.array([tfidf[dok][term] for term in vocab], dtype=float)
    skor_vsm[dok] = cosine_similarity_manual(vektor_query, vektor_dok)

ranking_vsm = pd.DataFrame({
    'Dokumen': nama_dokumen,
    'Skor VSM': [skor_vsm[d] for d in nama_dokumen]
}).sort_values('Skor VSM', ascending=False).reset_index(drop=True)

ranking_vsm['Ranking VSM'] = ranking_vsm.index + 1
display(ranking_vsm)

,Dokumen,Skor VSM,Ranking VSM
0,D1,0.312263,1
1,D3,0.312263,2
2,D2,0.000000,3
3,D4,0.000000,4
4,D5,0.000000,5


## Perbandingan Hasil Ranking

Tabel berikut menampilkan perbandingan ranking TF-IDF dan VSM secara berdampingan.

In [11]:
hasil_perbandingan = (
    ranking_tfidf[['Dokumen', 'Skor TF-IDF', 'Ranking TF-IDF']]
    .merge(ranking_vsm[['Dokumen', 'Skor VSM', 'Ranking VSM']], on='Dokumen', how='inner')
)

hasil_perbandingan = hasil_perbandingan.sort_values('Ranking TF-IDF').reset_index(drop=True)

display(hasil_perbandingan[['Dokumen', 'Skor TF-IDF', 'Ranking TF-IDF', 'Skor VSM', 'Ranking VSM']])

,Dokumen,Skor TF-IDF,Ranking TF-IDF,Skor VSM,Ranking VSM
0,D1,0.099485,1,0.312263,1
1,D3,0.099485,2,0.312263,2
2,D2,0.000000,3,0.000000,3
3,D4,0.000000,4,0.000000,4
4,D5,0.000000,5,0.000000,5


## Catatan Singkat: Boolean vs VSM

Pada **Boolean model**, pencarian menggunakan operator logika **AND**, **OR**, dan **NOT** untuk memfilter dokumen. Hasil akhirnya bersifat biner, yaitu dokumen dianggap relevan atau tidak relevan (0 atau 1). 

Pada **VSM + TF-IDF**, query dan dokumen diberi skor kemiripan kontinu (misalnya dari nilai rendah sampai tinggi). Karena skornya kontinu, dokumen bisa diurutkan (ranking) dari yang paling mirip hingga yang paling tidak mirip terhadap query.

## Analisis & Kesimpulan

Untuk query **"kecerdasan buatan pendidikan digital"**, urutan ranking TF-IDF dan VSM pada hasil saat ini **sama**, yaitu dokumen yang paling relevan tetap berada di posisi atas pada kedua metode. Kesamaan ini terjadi karena term-term query muncul pada dokumen yang serupa, sehingga pola bobot yang terbentuk juga mirip.

Secara konsep, hasil kedua metode bisa berbeda pada kasus lain. Jika ada dokumen yang panjang dan mengandung banyak term, pendekatan penjumlahan TF-IDF murni dapat memberi keuntungan lebih besar pada dokumen tersebut. Sebaliknya, VSM melakukan normalisasi dengan panjang vektor (norm), sehingga perbandingan antar dokumen menjadi lebih adil ketika panjang dokumen tidak sama.

Karena itu, TF-IDF penjumlahan cocok untuk melihat akumulasi bobot, sedangkan VSM lebih kuat untuk mengukur kemiripan ter-normalisasi antara query dan dokumen.

## Hasil perhitungan jarak dokumen dan query

Tabel berikut menampilkan komponen kuadrat vektor per token (`Q²`, `D1²`, dst), lalu norm vektor (`Sqrt(Q)` dan `Sqrt(Di)`).

In [7]:
# Komponen kuadrat vektor query dan dokumen
kolom_dokumen = nama_dokumen.copy()

baris_jarak = []
for idx, term in enumerate(vocab):
    item = {'Token': term, 'Q²': float(vektor_query[idx] ** 2)}
    for dok in kolom_dokumen:
        item[f'{dok}²'] = float(tfidf[dok][term] ** 2)
    baris_jarak.append(item)

df_jarak_komponen = pd.DataFrame(baris_jarak)

# Tampilkan token yang relevan (minimal satu nilai non-zero)
kolom_nilai = ['Q²'] + [f'{dok}²' for dok in kolom_dokumen]
mask_nonzero = (df_jarak_komponen[kolom_nilai].sum(axis=1) > 0)
df_jarak_tampil = df_jarak_komponen[mask_nonzero].copy().reset_index(drop=True)

# Baris norm sesuai rumus: sqrt(Σ komponen kuadrat)
sqrt_q = float(np.sqrt(df_jarak_komponen['Q²'].sum()))
sqrt_di = {
    dok: float(np.sqrt(df_jarak_komponen[f'{dok}²'].sum()))
    for dok in kolom_dokumen
}

baris_sqrt = {'Token': 'Sqrt'}
baris_sqrt['Q²'] = sqrt_q
for dok in kolom_dokumen:
    baris_sqrt[f'{dok}²'] = sqrt_di[dok]

df_hasil_jarak = pd.concat([
    df_jarak_tampil,
    pd.DataFrame([baris_sqrt])
], ignore_index=True)

print('Hasil perhitungan jarak dokumen dan query')
display(df_hasil_jarak)


Hasil perhitungan jarak dokumen dan query


,Token,Q²,D1²,D2²,D3²,D4²,D5²
0,abrasi,0.000000,0.000000,0.000000,0.000000,0.000000,0.004038
1,ada,0.000000,0.000000,0.000000,0.000000,0.004038,0.000000
2,ajar,0.000000,0.000000,0.000000,0.003393,0.000000,0.000000
3,anak,0.000000,0.000000,0.000000,0.000000,0.004038,0.000000
4,analisis,0.000000,0.003393,0.000000,0.000000,0.000000,0.000000
5,bantu,0.000000,0.000000,0.000000,0.003393,0.000000,0.000000
6,bas,0.000000,0.000000,0.000000,0.003393,0.000000,0.000000
7,bola,0.000000,0.000000,0.002493,0.000000,0.000000,0.000000
8,buat,0.017595,0.001100,0.000000,0.001100,0.000000,0.000000
9,cerdas,0.017595,0.001100,0.000000,0.001100,0.000000,0.000000
